In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
import config

# Verify data is accessible
try:
    config.assert_data_exists()
    print("\u2713 Data path:", config.DATA_ROOT)
except FileNotFoundError as e:
    print("\u2717 Data path error:", e)

print("Part 2 raw:", config.DATA_PART2_RAW)
print("Part 2 processed:", config.DATA_PART2_PROCESSED)

✓ Data path: /Users/hareee234/Library/CloudStorage/GoogleDrive-hareee234@gmail.com/My Drive/sem-8/CMPE188/flight-delay-proj-data
Part 2 raw: /Users/hareee234/Library/CloudStorage/GoogleDrive-hareee234@gmail.com/My Drive/sem-8/CMPE188/flight-delay-proj-data/part2/raw
Part 2 processed: /Users/hareee234/Library/CloudStorage/GoogleDrive-hareee234@gmail.com/My Drive/sem-8/CMPE188/flight-delay-proj-data/part2/processed


# Part 2 — 00: Data Load & Merge\n**CMPE 188 | Flight Delay Prediction — BTS 2023 Dataset**\n\nTables:\n- `US_flights_2023.csv` — main flights (1.15 GB)\n- `weather_meteo_by_airport.csv` — daily weather per airport (7.4 MB)\n- `airports_geolocation.csv` — airport metadata (27 KB)

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load all three tables using config helpers
flights = config.load_flights_2023()
weather = config.load_weather_meteo()
airports = config.load_airports_geo()

print("Flights:", flights.shape)
print("Weather:", weather.shape)
print("Airports:", airports.shape)

Flights: (6743404, 24)
Weather: (132860, 10)
Airports: (364, 7)


In [3]:
# Quick peek at flights columns
print("Flights columns:", flights.columns.tolist())
flights.head(3)

Flights columns: ['FlightDate', 'Day_Of_Week', 'Airline', 'Tail_Number', 'Dep_Airport', 'Dep_CityName', 'DepTime_label', 'Dep_Delay', 'Dep_Delay_Tag', 'Dep_Delay_Type', 'Arr_Airport', 'Arr_CityName', 'Arr_Delay', 'Arr_Delay_Type', 'Flight_Duration', 'Distance_type', 'Delay_Carrier', 'Delay_Weather', 'Delay_NAS', 'Delay_Security', 'Delay_LastAircraft', 'Manufacturer', 'Model', 'Aicraft_age']


,FlightDate,Day_Of_Week,Airline,Tail_Number,Dep_Airport,Dep_CityName,DepTime_label,Dep_Delay,Dep_Delay_Tag,Dep_Delay_Type,...,Flight_Duration,Distance_type,Delay_Carrier,Delay_Weather,Delay_NAS,Delay_Security,Delay_LastAircraft,Manufacturer,Model,Aicraft_age
0,2023-01-02,1,Endeavor Air,N605LR,BDL,"Hartford, CT",Morning,-3,0,Low <5min,...,56,Short Haul >1500Mi,0,0,0,0,0,CANADAIR REGIONAL JET,CRJ,16
1,2023-01-03,2,Endeavor Air,N605LR,BDL,"Hartford, CT",Morning,-5,0,Low <5min,...,62,Short Haul >1500Mi,0,0,0,0,0,CANADAIR REGIONAL JET,CRJ,16
2,2023-01-04,3,Endeavor Air,N331PQ,BDL,"Hartford, CT",Morning,-5,0,Low <5min,...,49,Short Haul >1500Mi,0,0,0,0,0,CANADAIR REGIONAL JET,CRJ,10


In [4]:
# Quick peek at weather columns
print("Weather columns:", weather.columns.tolist())
weather.head(3)

Weather columns: ['time', 'tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wdir', 'wspd', 'pres', 'airport_id']


,time,tavg,tmin,tmax,prcp,snow,wdir,wspd,pres,airport_id
0,2023-01-01,8.1,2.2,11.7,0.0,0.0,278.0,9.7,1013.8,ABE
1,2023-01-02,5.4,0.0,11.7,0.0,0.0,353.0,3.6,1019.6,ABE
2,2023-01-03,8.4,7.2,9.4,15.2,0.0,50.0,5.0,1013.9,ABE


In [5]:
# Quick peek at airports columns
print("Airports columns:", airports.columns.tolist())
airports.head(3)

Airports columns: ['IATA_CODE', 'AIRPORT', 'CITY', 'STATE', 'COUNTRY', 'LATITUDE', 'LONGITUDE']


,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE
0,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.44040
1,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.68190
2,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919


## 1. Validate Nulls & Coverage

In [6]:
# Flights null summary
null_counts = flights.isnull().sum()
print("Flights nulls:")
print(null_counts[null_counts > 0])

# Weather null summary
null_counts = weather.isnull().sum()
print("\nWeather nulls:")
print(null_counts[null_counts > 0])

# Airports null summary
null_counts = airports.isnull().sum()
print("\nAirports nulls:")
print(null_counts[null_counts > 0])

Flights nulls:
Series([], dtype: int64)

Weather nulls:
Series([], dtype: int64)

Airports nulls:
Series([], dtype: int64)


## 2. Prepare Weather for Join

In [7]:
# Weather is indexed by date + airport_id
weather['time'] = pd.to_datetime(weather['time'])
weather.head()

,time,tavg,tmin,tmax,prcp,snow,wdir,wspd,pres,airport_id
0,2023-01-01,8.1,2.2,11.7,0.0,0.0,278.0,9.7,1013.8,ABE
1,2023-01-02,5.4,0.0,11.7,0.0,0.0,353.0,3.6,1019.6,ABE
2,2023-01-03,8.4,7.2,9.4,15.2,0.0,50.0,5.0,1013.9,ABE
3,2023-01-04,11.1,6.7,17.2,0.0,0.0,302.0,4.7,1009.8,ABE
4,2023-01-05,12.7,6.7,14.4,7.9,0.0,292.0,7.2,1013.0,ABE


## 3. Prepare Flights for Join

In [8]:
# Parse FlightDate to datetime for joining with weather
flights['FlightDate'] = pd.to_datetime(flights['FlightDate'])

# Verify target variable distribution
print("Dep_Delay_Tag distribution:")
print(flights['Dep_Delay_Tag'].value_counts().sort_index())
print(f"Delay rate: {flights['Dep_Delay_Tag'].mean():.3f}")

Dep_Delay_Tag distribution:
Dep_Delay_Tag
0    4187645
1    2555759
Name: count, dtype: int64
Delay rate: 0.379


## 4. Join Weather (Departure Airport)

In [9]:
# Add departure weather
weather_dep = weather.add_prefix('dep_').rename(columns={'dep_time': 'FlightDate', 'dep_airport_id': 'Dep_Airport'})
flights = flights.merge(weather_dep, on=['FlightDate', 'Dep_Airport'], how='left')

# Add arrival weather (optional but useful)
weather_arr = weather.add_prefix('arr_').rename(columns={'arr_time': 'FlightDate', 'arr_airport_id': 'Arr_Airport'})
flights = flights.merge(weather_arr, on=['FlightDate', 'Arr_Airport'], how='left')

print(f"Shape after weather merge: {flights.shape}")
weather_cols = [c for c in flights.columns if c.startswith(('dep_', 'arr_'))]
print(f"Weather columns added: {len(weather_cols)}")
print(f"Rows with missing weather: {flights[weather_cols].isnull().any(axis=1).sum():,} / {len(flights):,}")

Shape after weather merge: (6743404, 40)
Weather columns added: 16
Rows with missing weather: 0 / 6,743,404


## 5. Join Airport Geolocation

In [10]:
# Join departure airport geolocation
airports_dep = airports.add_prefix('dep_').rename(columns={'dep_IATA_CODE': 'Dep_Airport'})
flights = flights.merge(airports_dep, on='Dep_Airport', how='left')

# Join arrival airport geolocation
airports_arr = airports.add_prefix('arr_').rename(columns={'arr_IATA_CODE': 'Arr_Airport'})
flights = flights.merge(airports_arr, on='Arr_Airport', how='left')

print(f"Shape after airport merge: {flights.shape}")

Shape after airport merge: (6743404, 52)


## 6. Handle Missing Values

In [11]:
# Fill missing weather with column means
for col in weather_cols:
    n_null = flights[col].isnull().sum()
    if n_null > 0:
        fill_val = flights[col].mean()
        flights[col] = flights[col].fillna(fill_val)
        print(f"  Filled {n_null:,} nulls in {col}")

# Verify
total_nulls = flights.isnull().sum().sum()
print(f"\nTotal nulls remaining: {total_nulls}")


Total nulls remaining: 0


## 7. Save Merged Dataset

In [12]:
# Ensure processed directory exists
config.DATA_PART2_PROCESSED.mkdir(parents=True, exist_ok=True)

out_path = config.DATA_PART2_PROCESSED / 'flights_2023_merged.csv'
flights.to_csv(out_path, index=False)

print(f"Saved → {out_path}")
print(f"Shape: {flights.shape}")
print(f"Columns: {len(flights.columns)}")
for col in flights.columns:
    print(f"  {col}")

Saved → /Users/hareee234/Library/CloudStorage/GoogleDrive-hareee234@gmail.com/My Drive/sem-8/CMPE188/flight-delay-proj-data/part2/processed/flights_2023_merged.csv
Shape: (6743404, 52)
Columns: 52
  FlightDate
  Day_Of_Week
  Airline
  Tail_Number
  Dep_Airport
  Dep_CityName
  DepTime_label
  Dep_Delay
  Dep_Delay_Tag
  Dep_Delay_Type
  Arr_Airport
  Arr_CityName
  Arr_Delay
  Arr_Delay_Type
  Flight_Duration
  Distance_type
  Delay_Carrier
  Delay_Weather
  Delay_NAS
  Delay_Security
  Delay_LastAircraft
  Manufacturer
  Model
  Aicraft_age
  dep_tavg
  dep_tmin
  dep_tmax
  dep_prcp
  dep_snow
  dep_wdir
  dep_wspd
  dep_pres
  arr_tavg
  arr_tmin
  arr_tmax
  arr_prcp
  arr_snow
  arr_wdir
  arr_wspd
  arr_pres
  dep_AIRPORT
  dep_CITY
  dep_STATE
  dep_COUNTRY
  dep_LATITUDE
  dep_LONGITUDE
  arr_AIRPORT
  arr_CITY
  arr_STATE
  arr_COUNTRY
  arr_LATITUDE
  arr_LONGITUDE
